# 📡 Notebook 7: Server-Side Push/Pull Mechanisms

So far, we've explored how clients can receive real-time updates (polling, SSE, WebSocket). But how does the server **itself** learn about changes? This notebook explores the "Hop 2" problem.

## Learning Objectives

By the end of this notebook, you'll understand:
- The "two hops" problem in real-time systems
- Database polling for change detection
- Redis Pub/Sub for server-to-server communication
- Change Data Capture (CDC) concepts
- When to use each approach

---

### 🔍 Open RedisInsight to Watch Pub/Sub Messages

1. Go to **http://localhost:5540**
2. Click "Add Redis Database"
3. Enter: Host = `redis`, Port = `6379`, Name = `realtime-redis`
4. Click "Add Redis Database"
5. Open the "Pub/Sub" tab to see messages flowing!

💡 **Tip**: Watch for messages on channels like `chat:general` when you run the Pub/Sub demos!

## 🔄 The Two Hops Problem

In a real-time system, updates need to travel two hops:

```
┌──────────┐     Hop 2      ┌──────────┐     Hop 1      ┌──────────┐
│  Source  │ ─────────────► │  Server  │ ─────────────► │  Client  │
│ (Events) │                │          │                │          │
└──────────┘                └──────────┘                └──────────┘

Examples of sources:
• User A sends a message → Server → User B's browser
• Driver updates location → Server → Passenger's phone
• Stock price changes → Server → Trader's dashboard
```

We've covered Hop 1 (Polling, Long Polling, SSE, WebSocket, WebRTC).

Now let's focus on **Hop 2**: How does the server know there's a new update?

## 📂 Approach 1: Pull via Database Polling

The simplest approach: store updates in a database, let clients poll for them.

```
┌────────┐     writes      ┌──────────┐     polls     ┌────────┐
│ User A │ ───────────────►│ Database │◄──────────────│ User B │
│(source)│                 │          │               │(client)│
└────────┘                 └──────────┘               └────────┘
```

The database acts as a **buffer** between the source and the consumer.

In [ ]:
# Example: Database-backed polling

from datetime import datetime
import time
import threading

# Simulate a database with a simple list
class SimpleDatabase:
    def __init__(self):
        self.messages = []
        self.lock = threading.Lock()
    
    def insert(self, message):
        """Insert a new message (source writes here)"""
        with self.lock:
            msg = {
                "id": len(self.messages) + 1,
                "text": message,
                "timestamp": datetime.now().timestamp()
            }
            self.messages.append(msg)
            return msg
    
    def query_since(self, since_timestamp):
        """Query messages after a timestamp (client polls here)"""
        with self.lock:
            return [m for m in self.messages if m["timestamp"] > since_timestamp]

# Create our "database"
db = SimpleDatabase()

print("📂 Database-backed Polling Example")
print("="*50)

# Source writes messages
def source_writes():
    for i in range(3):
        time.sleep(1)
        msg = db.insert(f"Message {i+1}")
        print(f"📤 Source wrote: {msg['text']}")

# Client polls for messages
received = []
polls = 0

def client_polls():
    global polls
    last_timestamp = datetime.now().timestamp()
    for _ in range(5):
        time.sleep(0.8)
        polls += 1
        new_messages = db.query_since(last_timestamp)
        if new_messages:
            for m in new_messages:
                print(f"📬 Client received: {m['text']}")
                received.append(m["text"])
                last_timestamp = max(last_timestamp, m['timestamp'])
        else:
            print("📭 Client poll: nothing new")

# Run both
source_thread = threading.Thread(target=source_writes)
client_thread = threading.Thread(target=client_polls)

source_thread.start()
client_thread.start()

source_thread.join()
client_thread.join()

print("\n✅ Database decouples source from client!")
print(f"   {polls} polls delivered {len(received)} messages "
      f"({polls - len(received)} of them returned nothing).")

# The cursor must deliver every write exactly once -- no gaps from a `>=`/`>`
# mix-up, no repeats from forgetting to advance `last_timestamp`.
assert received == ["Message 1", "Message 2", "Message 3"], (
    f"the `since` cursor lost or duplicated writes: {received}"
)

### Pros & Cons of Database Polling

**Pros:**
- 🟢 Very simple to implement
- 🟢 Source and client are completely decoupled
- 🟢 Messages are persisted (can replay)
- 🟢 Stateless servers

**Cons:**
- 🔴 Not real-time (latency = poll interval)
- 🔴 High DB load with frequent polling
- 🔴 Most polls return empty

## 🎯 Approach 2: Push via Consistent Hashing

For real-time updates with WebSocket/SSE, we need to PUSH to the right server. But which server holds User B's connection?

**Solution: Hash the user ID to determine the server!**

```
hash("userB") % num_servers = server_index
```

In [ ]:
# Demonstrate simple (modulo) hashing for server assignment.
#
# NOTE: we deliberately do NOT use Python's builtin hash() here. It is salted
# per process (PYTHONHASHSEED), so this cell would print different assignments
# every time you restart the kernel -- and any claim we made about the result
# would be luck. A stable digest makes the numbers reproducible.
import hashlib

def stable_hash(key: str) -> int:
    return int(hashlib.md5(key.encode()).hexdigest(), 16)

def simple_hash_assignment(user_id: str, num_servers: int) -> int:
    """Determine which server handles a user using simple modulo hashing."""
    return stable_hash(user_id) % num_servers

print("🎯 Simple Hash-based Server Assignment")
print("="*50)

users = ["alice", "bob", "charlie", "david", "eve"]

print("\nWith 3 servers:\n")
before = {u: simple_hash_assignment(u, 3) for u in users}
for user in users:
    print(f"  {user:10} → Server {before[user]}")

print("\n" + "="*50)
print("\n⚠️ Problem: What happens when we add a 4th server?\n")

after = {u: simple_hash_assignment(u, 4) for u in users}
for user in users:
    tag = "" if after[user] == before[user] else "  (MOVED)"
    print(f"  {user:10} → Server {after[user]}{tag}")

# Five names is far too small a sample to say anything about "almost all".
# Measure the churn over a realistic population instead of asserting it.
population = [f"user-{i}" for i in range(10_000)]
moved = sum(
    simple_hash_assignment(u, 3) != simple_hash_assignment(u, 4)
    for u in population
)
frac = moved / len(population)

print(f"\n📊 Over {len(population):,} users, 3 servers → 4 servers:")
print(f"   {moved:,} reassigned ({frac:.1%})")
print("\n😱 Modulo hashing reshuffles nearly everyone: `h % 3` and `h % 4` are")
print("   unrelated, so ~3 in 4 users land somewhere new. Every one of those")
print("   is a dropped WebSocket connection.")

# ~75% is the theoretical expectation (a key keeps its slot only when the two
# remainders coincide). Fail loudly if this ever stops being catastrophic.
assert frac > 0.6, f"expected modulo rehashing to move most keys, got {frac:.1%}"

SIMPLE_HASH_CHURN = frac   # notebook 07 compares against this below

### Consistent Hashing to the Rescue!

**Consistent hashing** minimizes reassignments when servers are added/removed.

In [ ]:
# Implement consistent hashing

import hashlib
from bisect import bisect_left

class ConsistentHashRing:
    """
    Consistent hash ring for server assignment.
    """
    
    def __init__(self, replicas: int = 100):
        self.replicas = replicas  # Virtual nodes per server
        self.ring: list[tuple[int, str]] = []   # sorted [(hash, server)]
        self._hashes: list[int] = []            # the hashes alone, for bisect
        self.servers = set()
    
    def _hash(self, key: str) -> int:
        """Hash a key to a position on the ring."""
        return int(hashlib.md5(key.encode()).hexdigest(), 16)
    
    def _reindex(self):
        self.ring.sort()
        # Cached so get_server() is O(log n). Rebuilding this list inside
        # get_server() would make every lookup O(n) -- fine for 7 users,
        # pathological for the 10,000-key measurement below.
        self._hashes = [h for h, _ in self.ring]
    
    def add_server(self, server: str):
        """Add a server to the ring."""
        self.servers.add(server)
        for i in range(self.replicas):
            self.ring.append((self._hash(f"{server}:{i}"), server))
        self._reindex()
    
    def remove_server(self, server: str):
        """Remove a server from the ring."""
        self.servers.discard(server)
        self.ring = [(h, s) for h, s in self.ring if s != server]
        self._reindex()
    
    def get_server(self, key: str) -> str | None:
        """Get the server for a given key: first ring point clockwise."""
        if not self.ring:
            return None
        idx = bisect_left(self._hashes, self._hash(key))
        if idx == len(self.ring):
            idx = 0            # wrap around the top of the ring
        return self.ring[idx][1]

# Demo consistent hashing
print("🎯 Consistent Hashing Demo")
print("="*50)

ring = ConsistentHashRing(replicas=50)
for server in ["server-1", "server-2", "server-3"]:
    ring.add_server(server)

users = ["alice", "bob", "charlie", "david", "eve", "frank", "grace"]

print("\nWith 3 servers:")
assignments_before = {u: ring.get_server(u) for u in users}
for user in users:
    print(f"  {user:10} → {assignments_before[user]}")

print("\n➕ Adding server-4...\n")

# Snapshot a big population BEFORE the change so we can measure real churn.
population = [f"user-{i}" for i in range(10_000)]
pop_before = {u: ring.get_server(u) for u in population}

ring.add_server("server-4")

print("With 4 servers:")
for user in users:
    server = ring.get_server(user)
    tag = "" if server == assignments_before[user] else "  (MOVED)"
    print(f"  {user:10} → {server}{tag}")

moved = sum(ring.get_server(u) != pop_before[u] for u in population)
frac = moved / len(population)

print(f"\n📊 Over the same {len(population):,} users, 3 servers → 4:")
print(f"   modulo hashing     : {SIMPLE_HASH_CHURN:>6.1%} reassigned")
print(f"   consistent hashing : {frac:>6.1%} reassigned")
print(f"\n✅ {SIMPLE_HASH_CHURN/frac:.1f}x fewer connections dropped.")
print("   Theory says ~1/4 should move when a 4th server joins: the new node")
print("   claims its share of the ring and NOBODY ELSE's assignment changes.")

# Two independent things must hold, and both are cheap to check:
#  1. it really is much better than modulo (that is the whole claim);
#  2. it moves roughly the theoretical 1/N share -- too little would mean the
#     new server is getting no traffic, too much means the ring is unbalanced.
assert frac < SIMPLE_HASH_CHURN / 2, (
    f"consistent hashing moved {frac:.1%} vs modulo's {SIMPLE_HASH_CHURN:.1%} -- "
    f"that is not an improvement worth the complexity"
)
assert 0.15 < frac < 0.40, (
    f"expected ~25% (=1/4) of keys to move to the new server, got {frac:.1%} -- "
    f"the ring is unbalanced (try more virtual nodes)"
)

# Load balance across the 4 servers -- the reason `replicas` exists at all.
from collections import Counter
spread = Counter(ring.get_server(u) for u in population)
print("\n📊 Load per server (why virtual nodes exist):")
for srv in sorted(spread):
    print(f"   {srv}: {spread[srv]:>5,} users ({spread[srv]/len(population):.1%})")

### How Consistent Hashing Enables Push

```
1. User B connects:
   hash("userB") → Server 2
   Server 2 stores the WebSocket connection

2. User A sends message to User B:
   API Server: hash("userB") → Server 2
   API Server: Send message to Server 2
   Server 2: Push to User B's WebSocket
```

In [ ]:
# Visualize the consistent hashing architecture

print("🏗️ Consistent Hashing Architecture")
print("="*60)
print("""
                    ┌─────────────────────┐
                    │   Coordination      │
                    │   (ZooKeeper/etcd)  │
                    │                     │
                    │ Stores: N servers,  │
                    │ their addresses     │
                    └──────────┬──────────┘
                               │
           ┌───────────────────┼───────────────────┐
           │                   │                   │
           ▼                   ▼                   ▼
    ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
    │  WS Server  │     │  WS Server  │     │  WS Server  │
    │      1      │     │      2      │     │      3      │
    └──────┬──────┘     └──────┬──────┘     └──────┬──────┘
           │                   │                   │
           │                   │                   │
    [Alice, Eve]         [Bob, David]       [Charlie]
    (connections)        (connections)      (connections)

Message Flow:
─────────────
1. Alice sends message to Bob
2. API Server: hash("bob") = Server 2
3. API Server → Server 2: "deliver to bob"
4. Server 2 → Bob's WebSocket: message!
""")

### Pros & Cons of Consistent Hashing

**Pros:**
- 🟢 Predictable server assignment
- 🟢 Minimal disruption when scaling
- 🟢 Direct routing to correct server

**Cons:**
- 🔴 Requires coordination service
- 🔴 Complex deployment/scaling logic
- 🔴 All servers need routing table
- 🔴 Connection state is lost on server failure

## 📨 Approach 3: Push via Pub/Sub

The most flexible approach: use a **Pub/Sub system** (like Redis) to broadcast updates!

```
┌────────┐     publish     ┌──────────┐     subscribe     ┌────────┐
│ Source │ ───────────────►│  Redis   │◄─────────────────│ Server │
│        │                 │  Pub/Sub │                   │        │
└────────┘                 └──────────┘                   └────────┘
```

---

### 🔍 Watch Pub/Sub in RedisInsight!

1. Open **http://localhost:5540**
2. Go to your database → **Pub/Sub** tab
3. Click **Subscribe** and enter `chat:*` to watch all chat channels
4. Run the cells below and watch messages flow in real-time!

In [ ]:
# Simulate Pub/Sub (without actual Redis)

import threading
from collections import defaultdict
from queue import Queue
import time

class SimplePubSub:
    """
    A simple in-memory Pub/Sub system.
    In production, use Redis Pub/Sub!
    """
    
    def __init__(self):
        self.subscribers = defaultdict(list)  # channel -> [queues]
        self.lock = threading.Lock()
    
    def subscribe(self, channel: str) -> Queue:
        """Subscribe to a channel. Returns a queue for receiving messages."""
        queue = Queue()
        with self.lock:
            self.subscribers[channel].append(queue)
        return queue
    
    def unsubscribe(self, channel: str, queue: Queue):
        """Unsubscribe from a channel."""
        with self.lock:
            if queue in self.subscribers[channel]:
                self.subscribers[channel].remove(queue)
    
    def publish(self, channel: str, message: dict):
        """Publish a message to all subscribers of a channel."""
        with self.lock:
            for queue in self.subscribers[channel]:
                queue.put(message)
        return len(self.subscribers[channel])

# Create our pub/sub system
pubsub = SimplePubSub()

print("📨 Pub/Sub Demo")
print("="*50)

In [ ]:
# Demo: Chat room with Pub/Sub

def simulate_ws_server(server_id: str, users: list, duration: float = 5):
    """
    Simulate a WebSocket server that subscribes to user channels.
    """
    print(f"🖥️ Server {server_id} starting with users: {users}")
    
    # Subscribe to each user's channel
    queues = {}
    delivered.setdefault(server_id, [])
    for user in users:
        channel = f"user:{user}"
        queues[user] = pubsub.subscribe(channel)
        print(f"   └─ Subscribed to {channel}")
    
    # Listen for messages
    start = time.time()
    while time.time() - start < duration:
        for user, queue in queues.items():
            if not queue.empty():
                msg = queue.get_nowait()
                delivered[server_id].append((user, msg["text"]))
                print(f"📬 Server {server_id}: Delivering to {user}: {msg}")
        time.sleep(0.1)
    
    # Cleanup
    for user in users:
        pubsub.unsubscribe(f"user:{user}", queues[user])
    
    print(f"🖥️ Server {server_id} stopping")

def send_message(from_user: str, to_user: str, text: str):
    """
    Send a message to a user via Pub/Sub.
    """
    channel = f"user:{to_user}"
    message = {
        "from": from_user,
        "text": text,
        "timestamp": time.time()
    }
    count = pubsub.publish(channel, message)
    print(f"📤 {from_user} → {to_user}: '{text}' (delivered to {count} subscriber(s))")

delivered: dict[str, list] = {}

# Start two servers with different users
server1_thread = threading.Thread(
    target=simulate_ws_server,
    args=("1", ["alice", "bob"], 4)
)
server2_thread = threading.Thread(
    target=simulate_ws_server,
    args=("2", ["charlie"], 4)
)

print("\n🚀 Starting servers...\n")
server1_thread.start()
server2_thread.start()

time.sleep(1)  # Let servers start

# Send some messages
print("\n📨 Sending messages...\n")
send_message("charlie", "alice", "Hello Alice!")
time.sleep(0.5)
send_message("alice", "bob", "Hey Bob!")
time.sleep(0.5)
send_message("bob", "charlie", "Hi Charlie!")

# Wait for servers to finish
server1_thread.join()
server2_thread.join()

print("\n✅ Demo complete!")
print("\n💡 Key insight: Servers don't need to know about each other!")
print("   Pub/Sub handles message routing automatically.")

# Each message must land on exactly the server that holds the recipient --
# that is the entire claim. A broadcast-to-everyone bug would still print
# something that looks fine.
assert delivered["1"] == [("alice", "Hello Alice!"), ("bob", "Hey Bob!")], (
    f"server 1 got the wrong messages: {delivered['1']}"
)
assert delivered["2"] == [("charlie", "Hi Charlie!")], (
    f"server 2 got the wrong messages: {delivered['2']}"
)
print("   ✅ Every message reached exactly the server holding the recipient.")

### 🔴 Real Redis Pub/Sub Demo

Now let's use **actual Redis** so you can see messages in RedisInsight!

In [ ]:
import json
import statistics
import threading
import time

import redis

r = redis.Redis(host='localhost', port=6379, decode_responses=True, socket_connect_timeout=3)

try:
    r.ping()
    REDIS_UP = True
    print("✅ Connected to Redis")
except redis.RedisError as exc:
    REDIS_UP = False
    print(f"⏭️  Redis is not reachable ({exc.__class__.__name__}).")
    print("   Start it with:  docker compose up -d   (from the lab directory)")
    print("   Skipping this cell -- the SimplePubSub demo above taught the same")
    print("   routing lesson without Redis.")

def redis_subscriber(channel_pattern: str, duration: float, sink: list):
    pubsub = r.pubsub()
    pubsub.psubscribe(channel_pattern)
    print(f"📡 Subscribed to: {channel_pattern}")

    # Use get_message() with a short timeout so the loop can exit
    # cleanly when `duration` elapses even if no messages arrive.
    # Using pubsub.listen() here would block forever on an idle channel.
    start = time.time()
    while time.time() - start < duration:
        message = pubsub.get_message(ignore_subscribe_messages=True, timeout=0.5)
        if message and message['type'] == 'pmessage':
            data = json.loads(message['data'])
            # Every payload carries the publish time, so we can measure the
            # end-to-end hop instead of quoting a number.
            sink.append((message['channel'], data, time.time() - data["sent_at"]))
            print(f"📬 Received on {message['channel']}: {data['text']!r} "
                  f"({(time.time() - data['sent_at'])*1000:.2f}ms after publish)")

    pubsub.punsubscribe(channel_pattern)
    pubsub.close()
    print("📡 Unsubscribed")

if REDIS_UP:
    got: list = []
    subscriber_thread = threading.Thread(target=redis_subscriber, args=("chat:*", 6, got))
    subscriber_thread.start()
    time.sleep(0.5)

    print("\n📤 Publishing messages to Redis...")
    print("   👀 Watch RedisInsight Pub/Sub tab!\n")

    def pub(channel, frm, text):
        r.publish(channel, json.dumps({"from": frm, "text": text, "sent_at": time.time()}))

    pub("chat:general", "alice", "Hello everyone!")
    time.sleep(1)
    pub("chat:general", "bob", "Hey Alice!")
    time.sleep(1)
    pub("chat:random", "charlie", "Anyone here?")

    subscriber_thread.join()

    hops = [lat for _, _, lat in got]
    print(f"\n✅ {len(got)}/3 messages received.")
    print(f"   Median publish→deliver hop: {statistics.median(hops)*1000:.2f}ms "
          f"(local Redis, same machine)")
    print("   Cross-AZ in a real deployment this is single-digit milliseconds --")
    print("   still small next to the polling interval it replaces.")

    assert len(got) == 3, f"psubscribe('chat:*') only saw {len(got)}/3 messages"
    assert {ch for ch, _, _ in got} == {"chat:general", "chat:random"}, (
        "the pattern subscription did not match both channels"
    )
    # The subscriber must be gone; a leaked pubsub connection is a real leak.
    assert not subscriber_thread.is_alive(), "subscriber thread did not stop"

### Pub/Sub Architecture

In [ ]:
# Visualize Pub/Sub architecture

print("🏗️ Pub/Sub Architecture")
print("="*60)
print("""
                    ┌─────────────────────┐
                    │   Load Balancer     │
                    │  (Least Connections)│
                    └──────────┬──────────┘
                               │
           ┌───────────────────┼───────────────────┐
           │                   │                   │
           ▼                   ▼                   ▼
    ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
    │  Endpoint   │     │  Endpoint   │     │  Endpoint   │
    │  Server 1   │     │  Server 2   │     │  Server 3   │
    └──────┬──────┘     └──────┬──────┘     └──────┬──────┘
           │                   │                   │
           │ subscribe         │ subscribe         │ subscribe
           │                   │                   │
           └───────────────────┼───────────────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │      Redis          │
                    │     Pub/Sub         │
                    └──────────┬──────────┘
                               │
                               │ publish
                               │
                    ┌──────────┴──────────┐
                    │   Message Sender    │
                    │   (API/Backend)     │
                    └─────────────────────┘

Flow:
─────
1. Client connects to ANY endpoint server
2. Endpoint server subscribes to user's channel on Redis
3. When message needs to be sent:
   - Backend publishes to Redis channel
   - Redis broadcasts to all subscribers
   - The right endpoint server receives it
   - Endpoint server pushes to client's WebSocket
""")

### Pros & Cons of Pub/Sub

**Pros:**
- 🟢 Endpoint servers are stateless (easy to scale)
- 🟢 No need for coordination service
- 🟢 Any server can handle any user
- 🟢 Simple load balancing

**Cons:**
- 🔴 Redis is single point of failure
- 🔴 Extra hop through Pub/Sub adds latency (sub-millisecond to local Redis — the cell above measures it — single-digit ms cross-AZ)
- 🔴 Doesn't know if user is actually connected
- 🔴 Many-to-many connections between servers and Pub/Sub

## 📊 Comparison: When to Use Each Approach

In [ ]:
# Comparison table

print("📊 Server-Side Push/Pull Comparison")
print("="*70)
print("""
                    DB Polling       Consistent Hash      Pub/Sub
────────────────────────────────────────────────────────────────────
Real-time?          ❌ No            ✅ Yes               ✅ Yes
Complexity          🟢 Low           🔴 High              🟡 Medium
Stateless servers   ✅ Yes           ❌ No (connections)  ✅ Yes
Scaling             🟢 Easy          🟡 Complex           🟢 Easy
Extra latency       Poll interval    ~0ms                 <1ms local, ~ms cross-AZ
Single point fail   DB               Coord service        Pub/Sub
Best for            Non-realtime     Heavy state          General use

────────────────────────────────────────────────────────────────────

DECISION GUIDE:
───────────────
• Need real-time? 
  No  → Database Polling ✅
  Yes → Continue...

• Have heavy per-connection state?
  Yes → Consistent Hashing ✅
  No  → Pub/Sub ✅

• Examples:
  - Email notifications → Database Polling
  - Google Docs (document state) → Consistent Hashing  
  - Chat application → Pub/Sub
""")

## 🎯 Interview Tips

In [ ]:
# Interview talking points

print("🎯 Interview Talking Points")
print("="*60)
print("""
THE TWO HOPS FRAMEWORK:
"When designing a real-time system, I think about two hops:
 1. How do updates get from the source to my server?
 2. How do updates get from my server to the client?"

FOR DATABASE POLLING:
"If updates don't need to be truly real-time, I'd use database
 polling. It's simple, and the DB naturally decouples producers
 from consumers. The trade-off is latency equals poll interval."

FOR CONSISTENT HASHING:
"For heavy state like collaborative editing, I'd use consistent
 hashing to route users to specific servers. This keeps document
 state local. I'd use ZooKeeper/etcd for coordination and minimize
 reconnection disruption during scaling."

FOR PUB/SUB:
"For most real-time apps like chat, I prefer Pub/Sub with Redis.
 Endpoint servers stay stateless - they just hold connections and
 subscribe to channels. When a message needs delivery, we publish
 to the user's channel and the right server receives it."

SCALING PUB/SUB:
"Redis Pub/Sub can become a bottleneck. I'd use Redis Cluster to
 shard channels across nodes. For very high scale, Kafka provides
 better durability and throughput."
""")

## 🧪 Quick Quiz

1. **You're building a notification system that needs to deliver alerts within 5 seconds. Database polling or Pub/Sub?**

2. **Why is consistent hashing better for Google Docs than Pub/Sub?**

3. **What's the main advantage of Pub/Sub over consistent hashing for a chat app?**

In [ ]:
# Quiz answers

print("📝 Quiz Answers")
print("="*50)
print("")
print("1. DATABASE POLLING would work!")
print("   5 seconds is long enough that you don't need")
print("   real-time infrastructure. Poll every 2-3 seconds.")
print("")
print("2. GOOGLE DOCS has heavy per-document state!")
print("   Each document needs to track operations, cursor")
print("   positions, and conflict resolution state.")
print("   Consistent hashing keeps this state on ONE server.")
print("")
print("3. STATELESS SERVERS!")
print("   With Pub/Sub, users can connect to ANY server.")
print("   Scaling is simple: just add servers.")
print("   No coordination service needed.")

## 🛟 Synthesis: Transport ≠ Durability

A nasty beginner mistake is to assume "real-time" implies "no messages lost."
It doesn't. Every transport we covered in this series answers a **different**
question: *how fast can bytes flow?* — not *what happens if the client was
offline when the event fired?*

The table below pairs each transport with the **recovery strategy** you need
to bolt on so users actually see missed events after a disconnect.

| Transport | Built-in recovery? | What you add for "no lost updates" |
|---|---|---|
| Simple polling | ✅ (implicit) | Client sends `since=<timestamp>` every poll — the DB is the source of truth. |
| Long polling | ✅ (implicit) | Same as above — client tracks last seen id. |
| SSE | ⚠️ Browser resends `Last-Event-ID` — **server must keep a replay buffer** (see NB 4). | Bounded buffer (e.g., last 100) + periodic reconcile for longer outages. |
| WebSocket | ❌ Nothing built in. | **Snapshot + sequence numbers**: on reconnect, send last seen seq → server replays from a log (Redis stream, Kafka, DB). |
| WebRTC data channel | ❌ Unreliable mode by design. | App-level retransmit / use the reliable mode for control messages. |
| Pub/Sub (Redis) | ❌ **Ephemeral** — messages published while no one was subscribed are **gone forever**. | Pair it with a persistent log (Redis Streams, Kafka) OR a DB write the client can query on reconnect. |
| Webhooks | ⚠️ Vendor retries for minutes-to-hours, then gives up. | **Nightly reconcile job** that pulls the vendor's API for anything your DB is missing. |

### The universal pattern: snapshot + stream

```
                   ┌────────────────────────┐
 Client reconnects │ 1. Fetch snapshot      │   ← what's the current state?
 ───────────────►  │    (REST / DB query)   │
                   └───────────┬────────────┘
                               │
                   ┌───────────▼────────────┐
                   │ 2. Open real-time feed │   ← WS / SSE, starting at
                   │    with a resume token │     the snapshot's sequence
                   └────────────────────────┘
```

Slack, Discord, Google Docs, Figma, trading platforms — **all** of them work
this way. "Real-time" is the fast path; the snapshot-and-replay fallback is
what makes the fast path *trustworthy*.

> 💡 Interview line: *"I'd separate the transport choice from the durability
> choice. WebSockets/SSE give me low latency; a persistent log plus a
> snapshot endpoint give me guaranteed delivery across reconnects."*


## 📚 Summary

### What We Learned:

1. **Two Hops** - Source→Server (Hop 2), Server→Client (Hop 1)

2. **Database Polling**
   - Simple, not real-time
   - Good for non-urgent updates

3. **Consistent Hashing**
   - Route users to specific servers
   - Good for heavy per-connection state
   - Complex scaling

4. **Pub/Sub (Redis)**
   - Stateless endpoint servers
   - Easy scaling
   - Good for most real-time apps

### The Complete Picture:

```
┌────────┐     Hop 2        ┌────────┐     Hop 1        ┌────────┐
│ Source │ ───────────────► │ Server │ ───────────────► │ Client │
└────────┘                  └────────┘                  └────────┘
              │                              │
              │                              │
     ┌────────┴────────┐          ┌─────────┴─────────┐
     │ DB Polling      │          │ Simple Polling    │
     │ Consistent Hash │          │ Long Polling      │
     │ Pub/Sub         │          │ SSE               │
     └─────────────────┘          │ WebSocket         │
                                  │ WebRTC            │
                                  └───────────────────┘
```

### 🎉 Congratulations!

You've completed the Real-time Updates pattern series! You now understand:
- All client-server protocols (Polling → WebRTC)
- Server-side update propagation strategies
- When to use each approach
- How to discuss these in interviews!